**Home Exercise 1 on Machine Translation**
Implement a `sequence2sequence` to **translate English to Vietnamese**. In this exercise, we will sequentially practice the steps to build a machine learning system for the machine translation task using a `seq2seq` model. These steps *include downloading and preprocessing bilingual data, creating training data, building a `seq2seq` model with attention, visualizing attention data, and translating new sentences on real-world data*.

Data: [IWSLT'15 English-Vietnamese](https://www.kaggle.com/datasets/tuannguyenvananh/iwslt15-englishvietnamese) (Train set: train.en and train.vi||| Val set: tst2012.en and tst2012.vi ||| Test set: tst2013.en and tst2013.vi).

In [12]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tqdm
import shutil, sys, zipfile
import random

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

from datetime import datetime
import datetime

print(f"The last time this notebook was run is: {datetime.datetime.now().strftime('%H:%M:%S %d/%m/%y')}")

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

The last time this notebook was run is: 00:20:58 14/12/25
Using device: cpu


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tuannguyenvananh/iwslt15-englishvietnamese")

print("Path to dataset files:", path)

ModuleNotFoundError: No module named 'kagglehub'

In [3]:
# Helper_functions
def unzip(path, dest, delete=True):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Zip file does not exist: {path}")
    
    if not zipfile.is_zipfile(path):
        raise zipfile.BadZipFile(f"Not a valid zip file: {path}")
    
    with zipfile.ZipFile(path, 'r') as zip_ref:
        zip_ref.extractall(dest)
        print(f"Unzipped into: {dest}")
    if delete:
        os.remove(path)
    else:
        print(f"Do not remove zipfile.")
    
    return dest

def move_path(src_path: str, dest_dir: str):
    if not os.path.exists(src_path):
        raise FileNotFoundError(f"Invalid {src_path}")
    
    os.makedirs(dest_dir, exist_ok=True)
    
    dst_path = os.path.join(dest_dir, os.path.basename(src_path))
    if os.path.exists(dst_path):
        print(f"Destination {dest_dir} is in used.")
        if os.path.isdir(dst_path):
            shutil.rmtree(dst_path)
        else:
            os.remove(dst_path)

    try:
        new_path = shutil.move(src_path, dest_dir)
        return new_path
    except Exception as e:
        if os.path.isdir(src_path):
            shutil.copytree(src_path, dst_path)
            shutil.rmtree(src_path)
            return dst_path
        else:
            raise e

In [4]:
!ls /home/dikhang/.cache/kagglehub/datasets/tuannguyenvananh/iwslt15-englishvietnamese/versions/1

In [6]:
src_dir = "/home/dikhang/.cache/kagglehub/datasets/tuannguyenvananh/iwslt15-englishvietnamese/versions/1"
filename = "IWSLT'15 en-vi"

data_dir = "./data"
full_path = os.path.join(src_dir, filename)
# file_path = move_path(full_path, data_dir)
files_path = os.path.join(data_dir, filename)
print("Moved to:", files_path)

Moved to: ./data/IWSLT'15 en-vi


In [7]:
!ls data/IWSLT\'15\ en-vi

dict.en-vi.txt		   train.vi.txt    tst2013.en.txt  vocab.vi.txt
luong-manning-iwslt15.pdf  tst2012.en.txt  tst2013.vi.txt
train.en.txt		   tst2012.vi.txt  vocab.en.txt


## Tokenizer

In [8]:
import os
import spacy
from collections import Counter

DATA_DIR = "data/IWSLT'15 en-vi"
FILES = {
    'train_src': os.path.join(DATA_DIR, 'train.en.txt'),
    'train_trg': os.path.join(DATA_DIR, 'train.vi.txt'),
    'val_src':   os.path.join(DATA_DIR, 'tst2012.en.txt'),
    'val_trg':   os.path.join(DATA_DIR, 'tst2012.vi.txt'),
    'test_src':  os.path.join(DATA_DIR, 'tst2013.en.txt'),
    'test_trg':  os.path.join(DATA_DIR, 'tst2013.vi.txt')
}

try:
    spacy_en = spacy.load('en_core_web_sm')
except OSError:
    print("Downloading en_core_web_sm...")
    from spacy.cli import download
    download("en_core_web_sm")
    spacy_en = spacy.load('en_core_web_sm')

def tokenize_en(text):
    return [tok.text.lower() for tok in spacy_en.tokenizer(text)]

def tokenize_vi(text):
    return text.lower().strip().split()


In [14]:
class Vocabulary:
    def __init__(self, freq_threshold=2):
        self.itos = {0: "<pad>", 1: "<sos>", 2: "<eos>", 3: "<unk>"}
        self.stoi = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
        self.freq_threshold = freq_threshold

    def __len__(self):
        return len(self.itos)

    def build_vocabulary(self, sentence_list, tokenizer):
        frequencies = Counter()
        idx = 4
        for sentence in sentence_list:
            for word in tokenizer(sentence):
                frequencies[word] += 1
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1
    
    def numericalize(self, text, tokenizer):
        tokenized_text = tokenizer(text)
        return [self.stoi.get(token, self.stoi["<unk>"]) for token in tokenized_text]

def read_lines(filepath):
    print(f"Reading: {filepath}")
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Not found: {filepath}")
    with open(filepath, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f.readlines()]

import yaml
def load_config(config_path = "config.yml"):
    with open(config_path, mode="r") as f:
        config = yaml.safe_load(f)
        
    return config

### Read data

In [10]:
train_en = read_lines(FILES['train_src'])
train_vi = read_lines(FILES['train_trg'])
val_en = read_lines(FILES['val_src'])
val_vi = read_lines(FILES['val_trg'])
test_en = read_lines(FILES['test_src'])
test_vi = read_lines(FILES['test_trg'])

print(f"\n--- Data Statistics ---")
print(f"Train size: {len(train_en)}")
print(f"Val size:   {len(val_en)}")
print(f"Test size:  {len(test_en)}")
print(f"Sample EN:  {train_en[0]}")
print(f"Sample VI:  {train_vi[0]}")

print("\nBuilding Vocabularies...")
vocab_en = Vocabulary(freq_threshold=2)
vocab_en.build_vocabulary(train_en, tokenize_en)

vocab_vi = Vocabulary(freq_threshold=2)
vocab_vi.build_vocabulary(train_vi, tokenize_vi)

print(f"EN Vocab: {len(vocab_en)} | VI Vocab: {len(vocab_vi)}")

Reading: data/IWSLT'15 en-vi/train.en.txt
Reading: data/IWSLT'15 en-vi/train.vi.txt
Reading: data/IWSLT'15 en-vi/tst2012.en.txt
Reading: data/IWSLT'15 en-vi/tst2012.vi.txt
Reading: data/IWSLT'15 en-vi/tst2013.en.txt
Reading: data/IWSLT'15 en-vi/tst2013.vi.txt

--- Data Statistics ---
Train size: 133317
Val size:   1553
Test size:  1268
Sample EN:  Rachel Pike : The science behind a climate headline
Sample VI:  Khoa học đằng sau một tiêu đề về khí hậu

Building Vocabularies...
EN Vocab: 28172 | VI Vocab: 12517


## Build the Data-loader class and Create training data

In [18]:
class NMTDataset(Dataset):
    def __init__(self, src_lines, trg_lines, src_vocab, trg_vocab, src_tok, trg_tok):
        self.src_lines = src_lines
        self.trg_lines = trg_lines
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab
        self.src_tok = src_tok
        self.trg_tok = trg_tok
        
    def __len__(self):
        return len(self.src_lines)
    
    def __getitem__(self, index):
        src_indexs = [1] + self.src_vocab.numericalize(self.src_lines[index], self.src_tok) + [2]
        trg_indexs = [1] + self.trg_vocab.numericalize(self.trg_lines[index], self.trg_tok) + [2]
        return torch.tensor(src_indexs), torch.tensor(trg_indexs)
    
def collate_function(batch):
    src_batch, trg_batch = zip(*batch)
    src_batch = pad_sequence(src_batch, padding_value=0, batch_first=False)
    trg_batch = pad_sequence(trg_batch, padding_value=0, batch_first=False)
    return src_batch, trg_batch

### Load config file

In [15]:
cfg = load_config()
print(f"Loaded config: {cfg}")

Loaded config: {'data': {'freq_threshold': 2, 'batch_size': 128}, 'model': {'enc_emb_dim': 256, 'dec_emb_dim': 256, 'enc_hid_dim': 512, 'dec_hid_dim': 512, 'enc_dropout': 0.5, 'dec_dropout': 0.5}, 'training': {'learning_rate': 0.001, 'n_epochs': 10, 'clip': 1.0, 'teacher_forcing_ratio': 0.5}}


In [19]:
BATCH_SIZE = cfg['data']['batch_size']

train_set = NMTDataset(train_en, train_vi, vocab_en, vocab_vi, tokenize_en, tokenize_vi)
val_set = NMTDataset(val_en, val_vi, vocab_en, vocab_vi, tokenize_en, tokenize_vi)
test_set = NMTDataset(test_en, test_vi, vocab_en, vocab_vi, tokenize_en, tokenize_vi)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_function)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_function)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_function)

## Build model

### Encoder class, attention class, decoder class

In [21]:
## encoder with GRU

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, enc_hid_dim, dec_hid_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, enc_hid_dim, bidirectional=True)
        self.fc = nn.Linear(enc_hid_dim*2, dec_hid_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, hidden = self.rnn(embedded)
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)))
        return outputs, hidden
    
class Attention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()
        self.attn = nn.Linear((enc_hid_dim * 2) + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)
        
    def forward(self, hidden, encoder_outputs):
        src_len = encoder_outputs.shape[0]
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2))) 
        attention = self.v(energy).squeeze(2)
        return F.softmax(attention, dim=1)

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hid_dim, dec_hid_dim, dropout, attention):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU((enc_hid_dim * 2) + emb_dim, dec_hid_dim)
        self.fc_out = nn.Linear((enc_hid_dim * 2) + dec_hid_dim + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, encoder_outputs):
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        a = self.attention(hidden, encoder_outputs).unsqueeze(1)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        weighted = torch.bmm(a, encoder_outputs).permute(1, 0, 2)
        rnn_input = torch.cat((embedded, weighted), dim=2)
        output, hidden = self.rnn(rnn_input, hidden.unsqueeze(0))
        prediction = self.fc_out(torch.cat((output.squeeze(0), weighted.squeeze(0), embedded.squeeze(0)), dim=1))
        return prediction, hidden.squeeze(0), a.squeeze(1)

### Seq2Seq class

In [22]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        encoder_outputs, hidden = self.encoder(src)
        input = trg[0,:]
        for t in range(1, trg_len):
            output, hidden, _ = self.decoder(input, hidden, encoder_outputs)
            outputs[t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1) 
            input = trg[t] if teacher_force else top1
        return outputs

## Training the model

### Configure training parameters

In [23]:
INPUT_DIM = len(vocab_en)
OUTPUT_DIM = len(vocab_vi)
ENC_HID_DIM = cfg['model']['enc_hid_dim']
DEC_HID_DIM = cfg['model']['dec_hid_dim']

ENC_EMB_DIM = cfg['model']['enc_emb_dim']
DEC_EMB_DIM = cfg['model']['enc_emb_dim']

ENC_DROPOUT = cfg['model']['enc_dropout']
DEC_DROPOUT = cfg['model']['dec_dropout']

LEARNING_RATE = cfg['training']['learning_rate']

attn = Attention(enc_hid_dim=ENC_HID_DIM, dec_hid_dim=DEC_HID_DIM)

enc = Encoder(INPUT_DIM, ENC_EMB_DIM, ENC_EMB_DIM, DEC_HID_DIM, ENC_DROPOUT)

dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, ENC_HID_DIM, DEC_HID_DIM, DEC_DROPOUT, attn)

model = Seq2Seq(enc, dec, device).to(device)

def init_weights(m):
    for name, param in m.named_parameters():
        if 'weight' in name:
            nn.init.normal_(param.data, mean=0, std=0.01)
        else:
            nn.init.constant_(param.data, 0)
            
model.apply(init_weights)

# Optimizer & Loss
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=vocab_vi.stoi["<pad>"])